# Actor-Critic Reference Tracking for the Unbalanced Disk

Assignment 4.2.2 — single policy that swings up and tracks a reference within ±15° of the upright.

All reward, environment and config logic lives in `a2c_reference_tracking_train.py`.
This notebook is a thin interface: import, train, evaluate, plot.

## 1. Setup

In [ ]:
import importlib
import json
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from stable_baselines3.common.vec_env import DummyVecEnv

import a2c_reference_tracking_train as ref_rl
importlib.reload(ref_rl)

# ── Editable run settings. Change only this line to switch algorithms. ──
ALGORITHM = ref_rl.ALGORITHM  # "A2C" or "PPO"

MODEL_PATH        = ref_rl.default_model_path(ALGORITHM)
REWARD_WEIGHTS    = ref_rl.REWARD_WEIGHTS.copy()
REWARD_PARAMS     = ref_rl.REWARD_PARAMS.copy()
TRAIN_HYPERPARAMS = ref_rl.default_hyperparams(ALGORITHM)
EPISODE_STEPS     = ref_rl.EPISODE_STEPS
SIMULATION_STEPS  = ref_rl.DEFAULT_SIMULATION_STEPS
SEED              = ref_rl.SEED
REF_RANGE_DEG     = ref_rl.REF_RANGE_DEG
OBS_NOISE_OMEGA   = ref_rl.OBS_NOISE_OMEGA

TRAIN_EPISODES = 1500   # more episodes — 5-dim obs is harder than 3-dim
TOTAL_TIMESTEPS = TRAIN_EPISODES * EPISODE_STEPS
ENTROPY_COEF   = 0.0005 if ALGORITHM.upper() == 'PPO' else 0.005
EVAL_EPISODES  = 20
DEVICE         = 'cpu'

print(f'Algorithm:       {ALGORITHM}')
print(f'Model path:      {MODEL_PATH}')
print(f'Reference range: ±{REF_RANGE_DEG}°')
print(f'Omega obs noise: {OBS_NOISE_OMEGA} rad/s')
print(f'Reward weights:  {REWARD_WEIGHTS}')
print(f'Reward params:   {REWARD_PARAMS}')
print(f'Hyperparams:     {TRAIN_HYPERPARAMS}')
print(f'Episodes:        {TRAIN_EPISODES} x {EPISODE_STEPS} = {TOTAL_TIMESTEPS:,} timesteps')
print(f'Eval episodes:   {EVAL_EPISODES}')
print(f'Seed:            {SEED}')
print(f'Device:          {DEVICE}')


## 2. Smoke Test

In [ ]:
# Verify env produces correct observation shape before training
test_env = ref_rl.make_env(SEED)()
obs, info = test_env.reset(seed=SEED)
next_obs, reward, _, _, info = test_env.step(np.array([0.0], dtype=np.float32))
print(f'Obs shape:       {obs.shape}  (expected (5,): sin θ, cos θ, ω, sin θ_ref, cos θ_ref)')
print(f'theta_ref:       {np.degrees(info["theta_ref"]):.1f}°')
print(f'obs:             {obs}')
print(f'training reward: {reward:.4f}')
test_env.close()


## 3. Train

In [ ]:
import hashlib
import shutil
from stable_baselines3.common.callbacks import EvalCallback

# Push notebook edits into the module globals used by envs, rewards, and build_model().
ref_rl.ALGORITHM = ALGORITHM
ref_rl.MODEL_PATH = MODEL_PATH
ref_rl.REWARD_WEIGHTS = REWARD_WEIGHTS.copy()
ref_rl.REWARD_PARAMS = REWARD_PARAMS.copy()
ref_rl.EPISODE_STEPS = EPISODE_STEPS
ref_rl.DEFAULT_SIMULATION_STEPS = SIMULATION_STEPS
ref_rl.SEED = SEED
ref_rl.REF_RANGE_DEG = REF_RANGE_DEG
ref_rl.REF_RANGE_RAD = np.deg2rad(REF_RANGE_DEG)
ref_rl.OBS_NOISE_OMEGA = OBS_NOISE_OMEGA
ref_rl.ROBUSTNESS['obs_noise_omega'] = OBS_NOISE_OMEGA
ref_rl.TRAIN_HYPERPARAMS = {
    **TRAIN_HYPERPARAMS,
    'seed': SEED,
    'episode_steps': EPISODE_STEPS,
}
TOTAL_TIMESTEPS = TRAIN_EPISODES * EPISODE_STEPS

best_model_dir = MODEL_PATH.parent / (MODEL_PATH.stem + "_best")
best_path = best_model_dir / 'best_model'
if best_model_dir.exists():
    shutil.rmtree(best_model_dir)
print(f'Refreshing best-model checkpoint in {best_model_dir}')
print(f'Active algorithm:      {ref_rl.ALGORITHM}')
print(f'Active omega obs noise:{OBS_NOISE_OMEGA} rad/s')
print(f'Active reward weights: {ref_rl.REWARD_WEIGHTS}')
print(f'Active reward params:  {ref_rl.REWARD_PARAMS}')
print(f'Active hyperparams:    {ref_rl.TRAIN_HYPERPARAMS}')
print(f'Active entropy coef:   {ENTROPY_COEF}')
print(f'Active eval episodes:  {EVAL_EPISODES}')
print(f'Active seed:           {ref_rl.SEED}')

train_env = DummyVecEnv([ref_rl.make_env(SEED, EPISODE_STEPS, OBS_NOISE_OMEGA)])
eval_env  = DummyVecEnv([ref_rl.make_env(SEED + 99, EPISODE_STEPS, 0.0)])
model = ref_rl.build_model(train_env, DEVICE, ENTROPY_COEF)

# Save the best model automatically during this training run — not just the final one.
# Policy degradation is common in on-policy RL: more episodes ≠ better policy.
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=str(best_model_dir),
    log_path=None,
    eval_freq=EPISODE_STEPS * 50,   # evaluate every 50 episodes
    n_eval_episodes=EVAL_EPISODES,
    deterministic=True,
    verbose=0,
)

t0 = time.perf_counter()
model.learn(total_timesteps=TOTAL_TIMESTEPS, callback=eval_callback, progress_bar=False)
elapsed = time.perf_counter() - t0

# Load the best model found during this run (not necessarily the last)
checkpoint_zip = best_path.with_suffix('.zip')
if not checkpoint_zip.exists():
    raise FileNotFoundError(f'EvalCallback did not create a fresh checkpoint at {checkpoint_zip}')
model = load_model_with_numpy_fallback(best_path, ref_rl.ALGORITHM, DEVICE)
model.save(MODEL_PATH)   # save this run's best as the final named model

checkpoint_sha1 = hashlib.sha1(checkpoint_zip.read_bytes()).hexdigest()[:12]
train_env.close(); eval_env.close()
print(f'Training time: {elapsed:.1f}s ({TOTAL_TIMESTEPS / elapsed:.1f} steps/s)')
print(f'Best model refreshed from {checkpoint_zip} and saved as {MODEL_PATH}.zip')
print(f'Fresh checkpoint SHA1: {checkpoint_sha1}')


## 4. Evaluate — Upright (θ_ref = 0°)

In [ ]:
def print_summary(label, rewards, actions, infos):
    ref_errors     = np.array([i['ref_error']     for i in infos])
    upright_errors = np.array([i['upright_error'] for i in infos])
    omegas         = np.array([i['omega']         for i in infos])
    env_rewards    = np.array([i['env_reward']    for i in infos])
    theta_ref_deg  = np.degrees(infos[0]['theta_ref'])
    print(f'=== {label} (θ_ref = {theta_ref_deg:.1f}°) ===')
    print(f'  steps:               {len(rewards)}')
    print(f'  external return:     {rewards.sum():.2f}')
    print(f'  mean ref error:      {ref_errors.mean():.4f} rad  ({np.degrees(ref_errors.mean()):.2f}°)')
    print(f'  final ref error:     {ref_errors[-1]:.4f} rad  ({np.degrees(ref_errors[-1]):.2f}°)')
    print(f'  mean upright error:  {upright_errors.mean():.4f} rad  ({np.degrees(upright_errors.mean()):.2f}°)')
    print(f'  max |omega|:         {np.abs(omegas).max():.2f} rad/s')
    print(f'  max |u|:             {np.abs(actions).max():.2f} V')

rewards0, actions0, infos0 = ref_rl.rollout(model, EPISODE_STEPS, theta_ref_deg=0.0)
print_summary('upright', rewards0, actions0, infos0)

# Save config after first evaluation
rollout_summary = {
    'steps':                  len(rewards0),
    'external_return':        round(float(rewards0.sum()), 2),
    'mean_ref_error_deg':     round(float(np.degrees(np.array([i['ref_error'] for i in infos0]).mean())), 3),
    'final_ref_error_deg':    round(float(np.degrees(infos0[-1]['ref_error'])), 3),
    'max_omega_rad_s':        round(float(np.abs(np.array([i['omega'] for i in infos0])).max()), 3),
    'max_action_V':           round(float(np.abs(actions0).max()), 3),
}
config_path = ref_rl.save_config(
    MODEL_PATH, entropy_coef=ENTROPY_COEF,
    total_timesteps=TOTAL_TIMESTEPS, training_time_s=elapsed,
    rollout_summary=rollout_summary,
)
print(f'Saved config: {config_path}')


## 5. Evaluate — ±15° References

In [ ]:
REFERENCE_DEGS = [-15.0, -10.0, -5.0, 0.0, 5.0, 10.0, 15.0]

rollouts = {}
for ref_deg in REFERENCE_DEGS:
    rewards, actions, infos = ref_rl.rollout(model, EPISODE_STEPS, theta_ref_deg=ref_deg)
    rollouts[ref_deg] = {
        'rewards': rewards,
        'actions': actions,
        'infos': infos,
    }
    print_summary(f'{ref_deg:+.0f}° reference', rewards, actions, infos)
    print()

# Backward-compatible aliases for the config cell and older plotting snippets.
rewards0 = rollouts[0.0]['rewards']
actions0 = rollouts[0.0]['actions']
infos0 = rollouts[0.0]['infos']


## 6. Plot — Compare Three References

In [ ]:
def rollout_arrays(data):
    infos = data['infos']
    rewards = data['rewards']
    actions = data['actions']
    theta_ref = infos[0]['theta_ref']
    final_theta = infos[-1]['theta_wrapped']
    return {
        't': np.arange(len(rewards)) * 0.025,
        'angle_deg': np.degrees(np.array([i['theta_wrapped'] for i in infos])),
        'ref_error_deg': np.degrees(np.array([i['ref_error'] for i in infos])),
        'omega': np.array([i['omega'] for i in infos]),
        'actions': actions,
        'rewards': rewards,
        'theta_ref_deg': np.degrees(ref_rl.wrap_angle(theta_ref)),
        'final_abs_error_deg': np.degrees(infos[-1]['ref_error']),
        'final_signed_error_deg': np.degrees(ref_rl.wrap_angle(final_theta - theta_ref)),
        'final_offset_deg': np.degrees(ref_rl.wrap_angle(final_theta - np.pi)),
    }

plot_data = {ref_deg: rollout_arrays(data) for ref_deg, data in rollouts.items()}
colors = plt.cm.coolwarm(np.linspace(0.05, 0.95, len(REFERENCE_DEGS)))
color_by_ref = dict(zip(REFERENCE_DEGS, colors))

# Previous detailed view for the three headline references.
def plot_detailed_rollout(axs, ref_deg, color):
    data = plot_data[ref_deg]
    label = f'{ref_deg:+.0f}°'
    axs[0].plot(data['t'], data['angle_deg'], color=color, label=label)
    axs[0].axhline(data['theta_ref_deg'], color=color, ls='--', lw=1, alpha=0.65)
    axs[1].plot(data['t'], data['ref_error_deg'], color=color, label=label)
    axs[2].plot(data['t'], data['omega'], color=color, label=label)
    axs[3].plot(data['t'], data['actions'], color=color, label=label)
    axs[4].plot(data['t'], data['rewards'], color=color, label=label)

fig_detail, detail_axs = plt.subplots(5, 1, figsize=(11, 10), sharex=True)
for ref_deg in [-15.0, 0.0, 15.0]:
    plot_detailed_rollout(detail_axs, ref_deg, color_by_ref[ref_deg])

detail_axs[0].set_ylabel('angle [deg]')
detail_axs[0].set_title(f'{ALGORITHM} Reference Tracking — Detailed View')
detail_axs[0].legend(title='target offset', loc='upper right')
detail_axs[1].set_ylabel('|error| [deg]')
detail_axs[2].set_ylabel('ω [rad/s]')
detail_axs[3].set_ylabel('u [V]'); detail_axs[3].set_ylim(-3.2, 3.2)
detail_axs[4].set_ylabel('reward'); detail_axs[4].set_xlabel('time [s]')
for ax in detail_axs:
    ax.grid(alpha=0.3)
fig_detail.tight_layout()
plt.show()

# Compact comparison across every tested reference.
fig, axs = plt.subplots(2, 2, figsize=(13, 8.5))
ax_angle, ax_err, ax_final, ax_actual = axs.ravel()

summary_rows = []
for ref_deg in REFERENCE_DEGS:
    data = plot_data[ref_deg]
    color = color_by_ref[ref_deg]
    label = f'{ref_deg:+.0f}°'
    summary_rows.append({
        'ref_deg': ref_deg,
        'final_abs_error_deg': float(data['final_abs_error_deg']),
        'final_signed_error_deg': float(data['final_signed_error_deg']),
        'final_offset_deg': float(data['final_offset_deg']),
    })

    ax_angle.plot(data['t'], data['angle_deg'], color=color, lw=1.8, label=label)
    ax_angle.axhline(data['theta_ref_deg'], color=color, ls='--', lw=0.8, alpha=0.55)
    ax_err.plot(data['t'], data['ref_error_deg'], color=color, lw=1.8, label=label)

summary = {key: np.array([row[key] for row in summary_rows]) for key in summary_rows[0]}
bar_colors = [color_by_ref[ref_deg] for ref_deg in REFERENCE_DEGS]
ax_final.bar(summary['ref_deg'], summary['final_abs_error_deg'], width=3.0, color=bar_colors, edgecolor='white')
for ref_deg, err in zip(summary['ref_deg'], summary['final_abs_error_deg']):
    ax_final.annotate(f'{err:.2f}', (ref_deg, err), textcoords='offset points', xytext=(0, 6), ha='center', fontsize=8)

ax_actual.plot([-15, 15], [-15, 15], color='#777777', ls='--', lw=1, label='ideal')
ax_actual.scatter(summary['ref_deg'], summary['final_offset_deg'], c=bar_colors, s=70, edgecolor='white', zorder=3, label='final')
for ref_deg, actual_deg, signed_err in zip(summary['ref_deg'], summary['final_offset_deg'], summary['final_signed_error_deg']):
    ax_actual.annotate(f'{signed_err:+.2f}°', (ref_deg, actual_deg), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

ax_angle.set_title(f'{ALGORITHM} Angle Tracking — All References')
ax_angle.set_ylabel('angle [deg]')
ax_angle.legend(title='target offset', ncol=2, fontsize=8)

ax_err.set_title('Absolute Reference Error Over Time')
ax_err.set_ylabel('|error| [deg]')
ax_err.set_xlabel('time [s]')
ax_err.set_ylim(bottom=0)

ax_final.set_title('Final Absolute Error by Reference')
ax_final.set_xlabel('target offset [deg]')
ax_final.set_ylabel('final |error| [deg]')
ax_final.set_xticks(REFERENCE_DEGS)

ax_actual.set_title('Final Achieved Offset vs Target')
ax_actual.set_xlabel('target offset [deg]')
ax_actual.set_ylabel('final achieved offset [deg]')
ax_actual.set_xticks(REFERENCE_DEGS)
ax_actual.set_aspect('equal', adjustable='box')
ax_actual.legend(fontsize=8)

for ax in axs.ravel():
    ax.grid(alpha=0.3)

fig.tight_layout()
plt.show()


## 7. Render Simulation

In [ ]:
# Change theta_ref_deg to test any target in [-15, +15]
TARGET_DEG = 0.0

sim_rewards, sim_actions, sim_infos = ref_rl.rollout(
    model,
    max_steps=SIMULATION_STEPS,
    theta_ref_deg=TARGET_DEG,
    deterministic=True,
    render=True,
)
print_summary(f'rendered simulation', sim_rewards, sim_actions, sim_infos)


In [ ]:
from collections import deque
from stable_baselines3 import A2C, PPO

CONFIG_DIR = Path("ref_models/cfg")


def ensure_numpy_load_compatibility():
    try:
        import numpy._core.numeric  # noqa: F401
    except ModuleNotFoundError:
        import sys
        import numpy.core as numpy_core
        import numpy.core.numeric as numpy_core_numeric
        import numpy.core.multiarray as numpy_core_multiarray
        import numpy.core.umath as numpy_core_umath

        sys.modules.setdefault("numpy._core", numpy_core)
        sys.modules.setdefault("numpy._core.numeric", numpy_core_numeric)
        sys.modules.setdefault("numpy._core.multiarray", numpy_core_multiarray)
        sys.modules.setdefault("numpy._core.umath", numpy_core_umath)


def load_saved_config(model_path):
    model_path = Path(model_path)
    candidates = [
        Path(str(model_path) + "_config.json"),
        CONFIG_DIR / f"{model_path.stem}_config.json",
    ]
    for candidate in candidates:
        if candidate.exists():
            with open(candidate, "r") as f:
                return json.load(f), candidate
    raise FileNotFoundError(f"No config file found for {model_path}. Checked: {candidates}")


def apply_saved_config(config, algorithm):
    ref_rl.ALGORITHM = algorithm.upper()
    ref_rl.REWARD_WEIGHTS = config["reward_weights"].copy()
    ref_rl.REWARD_PARAMS = config["reward_params"].copy()
    ref_rl.REF_RANGE_DEG = float(config["ref_range_deg"])
    ref_rl.REF_RANGE_RAD = np.deg2rad(ref_rl.REF_RANGE_DEG)
    ref_rl.ROBUSTNESS = config.get("robustness", ref_rl.ROBUSTNESS).copy()
    ref_rl.OBS_NOISE_OMEGA = float(ref_rl.ROBUSTNESS.get("obs_noise_omega", 0.0))


def load_model_with_numpy_fallback(model_path, algorithm, device):
    ensure_numpy_load_compatibility()
    model_class = PPO if algorithm.upper() == "PPO" else A2C
    tmp_env = ref_rl.DiskRefTrackEnv(max_episode_steps=EPISODE_STEPS, obs_noise_omega=0.0)
    try:
        custom_objects = {
            "observation_space": tmp_env.observation_space,
            "action_space": tmp_env.action_space,
            "_last_obs": None,
            "_last_episode_starts": None,
            "ep_info_buffer": deque(maxlen=100),
            "ep_success_buffer": deque(maxlen=100),
        }
        return model_class.load(model_path, device=device, custom_objects=custom_objects)
    finally:
        tmp_env.close()


def print_saved_config_summary(config, config_path):
    print(f"  config file:    {config_path}")
    print(f"  saved at:       {config.get('saved_at', '?')}")
    print(f"  model:          {config.get('model_path', '?')}")
    print(f"  algorithm:      {config.get('algorithm', ALGORITHM)}")
    print(f"  ref range:      ±{config.get('ref_range_deg', '?')}°")
    print(f"  reward weights: {config['reward_weights']}")
    print(f"  reward params:  {config['reward_params']}")
    print(f"  robustness:     {config.get('robustness', {})}")
    if "training" in config:
        t = config["training"]
        print(f"  training:       {t['total_timesteps']:,} steps in {t['training_time_s']}s")
    if "rollout_summary" in config:
        print(f"  rollout:        {config['rollout_summary']}")


## 8. Load and inspect a saved model

This cell loads one saved reference-tracking policy and its matching configuration. It is useful for checking an existing A2C or PPO model without retraining.


In [ ]:
# Choose which saved reference-tracking policy to inspect.
LOAD_ALGORITHM = ALGORITHM
LOAD_MODEL_PATH = MODEL_PATH

saved_config, saved_config_path = load_saved_config(LOAD_MODEL_PATH)
LOAD_ALGORITHM = saved_config.get("algorithm", LOAD_ALGORITHM).upper()
apply_saved_config(saved_config, LOAD_ALGORITHM)

model = load_model_with_numpy_fallback(LOAD_MODEL_PATH, LOAD_ALGORITHM, DEVICE)

print("=== Loaded model ===")
print(f"  model file:     {LOAD_MODEL_PATH}.zip")
print_saved_config_summary(saved_config, saved_config_path)

rewards0, actions0, infos0 = ref_rl.rollout(model, EPISODE_STEPS, theta_ref_deg=0.0)
print_summary("loaded model, upright reference", rewards0, actions0, infos0)
